In [1]:

from pyspark.sql import functions as F
from pyspark.sql.types import *


In [20]:

storage_account = "staccpakwheels"
container = "landing"
#folderdata = "pakwheelsdata"

raw_path  = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pakwheelsdata/"
good_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/pakwheelsgooddata/"


In [26]:

correct_columns = [
'MakeModel',
 'Year',
 'Mileage',
 'FuelType',
 'Transmission',
 'City',
 'Colour',
 'ManufactureType',
 'EngineCapacity',
 'CarType',
 'PostLastUpdatedat',
 'postId',
 'CarAddress',
 'Demand',
 'Alloy Wheels',
 'Front Fog Lights',
 'DRLs',
 'Sun Roof',
 'ABS',
 'Air Bags',
 'Air Conditioning',
 'Alloy Rims',
 'Infotainment System',
 'Front Speakers',
 'Rear Speakers',
 'Steering Switches',
 'Paddle Shifters',
 'Traction Control',
 'Front Camera',
 'Rear Camera',
 'Immobilizer Key',
 'Power Locks',
 'Power Steering',
 'Cool Box',
 'Cruise Control'
]


In [27]:
df_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")     # ✅ fixes broken rows
    .option("quote", '"')
    .option("escape", "\\")
    .option("delimiter", ",")
    .option("mode", "PERMISSIVE")
    .csv(raw_path)
    .toDF(*correct_columns)
)


In [28]:
df_raw['PostLastUpdatedat','CarAddress','Demand','Alloy Wheels','Front Fog Lights'].show(1,False)

In [29]:
df_raw.columns

In [30]:

rename_map = {
    "Alloy Wheels": "AlloyWheels",
    "Front Fog Lights": "FrontFogLights",
    "Sun Roof": "SunRoof",
    "Air Bags": "AirBags",
    "Air Conditioning": "AirConditioning",
    "Alloy Rims": "AlloyRims",
    "Infotainment System": "InfotainmentSystem",
    "Front Speakers": "FrontSpeakers",
    "Rear Speakers": "RearSpeakers",
    "Steering Switches": "SteeringSwitches",
    "Paddle Shifters": "PaddleShifters",
    "Traction Control": "TractionControl",
    "Front Camera": "FrontCamera",
    "Rear Camera": "RearCamera",
    "Immobilizer Key": "ImmobilizerKey",
    "Power Locks": "PowerLocks",
    "Power Steering": "PowerSteering",
    "Cool Box": "CoolBox",
    "Cruise Control": "CruiseControl"
}

for old_name, new_name in rename_map.items():
    if old_name in df_raw.columns:
        df_raw = df_raw.withColumnRenamed(old_name, new_name)



In [31]:
df_raw.columns

In [33]:

df_clean = (
    df_raw
    .withColumn("Year", F.col("Year").cast("int"))
    .withColumn("PostLastUpdatedat", F.to_date("PostLastUpdatedat", "MMM dd, yyyy"))
    .withColumn("Mileage", F.regexp_replace("Mileage", ",", "")
)


In [34]:
df_clean.show(1,False)

In [36]:

df_clean = (
    df_clean
    .withColumn(
        "Mileage",
        F.regexp_extract("Mileage", r"(\d+)", 1).cast("int")
    )
    .withColumn(
        "EngineCapacity",
        F.regexp_extract("EngineCapacity", r"(\d+)", 1).cast("int")
    )
    .withColumn(
        "Demand",
        F.regexp_replace(F.col("Demand"), r"PKR\s*", "")
    )
    .withColumn(
        "DemandInLacs",
        F.regexp_extract("Demand", r"(\d+)", 1).cast("int")
    )
)






In [37]:
for col_name, dtype in df_clean.dtypes:
    if dtype == "string":
        df_clean = df_clean.withColumn(col_name, F.trim(F.col(col_name)))

In [38]:

df_clean.select(
    "Mileage",
    "EngineCapacity",
    "AlloyWheels"
).show(5, truncate=False)


In [39]:
df_clean.show(1,False)

In [53]:

from pyspark.sql.functions import year, month, dayofweek, datediff, current_date,when



df_clean = (
    df_clean
    .withColumn("PostYear", year("PostLastUpdatedat"))
    .withColumn("PostMonth", month("PostLastUpdatedat"))
    .withColumn("PostDayOfWeek", dayofweek("PostLastUpdatedat"))
    .withColumn("ListingAgeDays", datediff(current_date(), "PostLastUpdatedat")
    

    )
    
)


In [51]:

df_clean = (
    df_clean
    .withColumn("DemandPKR", df_clean.DemandLacs * 100000)
    .withColumn("CarAge", year(current_date()) - df_clean.Year)
    
    
    
)
#.withColumn("PricePerCC", df_clean.DemandPKR / df_clean.EngineCapacity)
#.withColumn("PricePerYearAge", df_clean.DemandPKR / (df_clean.CarAge + 1)


In [54]:

df_clean = df_clean.withColumn(
    "PriceCategory",
    when(df_clean.DemandLacs < 10, "Budget")
    .when(df_clean.DemandLacs < 30, "Mid")
    .otherwise("Premium")
)


In [55]:

df_clean = (
    df_clean
    .withColumn("MileagePerYear", df_clean.Mileage / (df_clean.CarAge + 1))
    .withColumn(
        "MileageBucket",
        when(df_clean.Mileage < 50000, "Low")
        .when(df_clean.Mileage < 100000, "Medium")
        .otherwise("High")
    )
)


In [57]:

df_clean = df_clean.withColumn(
    "SafetyScore",
    df_clean.ABS + df_clean.AirBags + df_clean.TractionControl
)

df_clean = df_clean.withColumn(
    "ComfortScore",
    df_clean.AirConditioning  + df_clean.CruiseControl + df_clean.PowerSteering
)

df_clean = df_clean.withColumn(
    "TechScore",
    df_clean.InfotainmentSystem + df_clean.RearCamera + df_clean.FrontCamera
)


In [58]:

major_cities = ["Karachi", "Lahore", "Islamabad"]

df_clean = df_clean.withColumn(
    "CityGroup",
    when(df_clean.City.isin(major_cities), df_clean.City).otherwise("Other")
)


In [59]:

df_clean = df_clean.withColumn(
    "Brand",
    F.split(df_clean.MakeModel, " ").getItem(0)
)


In [60]:

df_clean = df_clean.withColumn(
    "MissingCriticalInfo",
    when(
        df_clean.Mileage.isNull() | df_clean.DemandLacs.isNull() | df_clean.EngineCapacity.isNull(),
        1
    ).otherwise(0)
)


In [61]:
df_clean.show(2,False)

In [62]:
(
    df_clean
    .write
    .mode("append")
    .parquet(good_path)
)

